In [1]:
# This notebook runs the full road damage detection workflow.
import cv2
import os
import numpy as np
import sys

# ---------------------------------------------------------
# 1. SETUP DIRECTORIES & CONFIGURATION
# ---------------------------------------------------------
# Use the current project folder.
base_dir = os.getcwd()
input_video_path = os.path.join(base_dir, "input_video", "video.mp4")

extract_frame_dir = os.path.join(base_dir, 'extract_frame')
enhanced_frame_dir = os.path.join(base_dir, 'enhanced_frames')
segmented_frame_dir = os.path.join(base_dir, 'segmented_frames')
detected_frame_dir = os.path.join(base_dir, 'detected_frames')
final_frame_dir = os.path.join(base_dir, 'final_frames')

output_video_path = os.path.join(base_dir, 'Final_Road_Damage_Detection.mp4')
enhanced_video_path = os.path.join(base_dir, 'enhanced_video.mp4')
segmented_video_path = os.path.join(base_dir, 'segmented_video.mp4')
detected_video_path = os.path.join(base_dir, 'detected_video.mp4')

# Create all output folders.
for folder in [extract_frame_dir, enhanced_frame_dir, segmented_frame_dir, detected_frame_dir, final_frame_dir]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# ---------------------------------------------------------
# 2. VIDEO FRAME EXTRACTION
# ---------------------------------------------------------
# Open the source video.
cap = cv2.VideoCapture(input_video_path)
frame_count = 0

# Extract every frame from the video.
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Optional: If you want to force all video to 1080p for speed, uncomment the line below:
    # frame = cv2.resize(frame, (1920, 1080))
    
    frame_name = os.path.join(extract_frame_dir, f"frame_{frame_count:05d}.jpg")
    cv2.imwrite(frame_name, frame)
    frame_count += 1
cap.release()
print(f"Extracted {frame_count} frames.")

# ---------------------------------------------------------
# 3. FRAME ENHANCEMENT FUNCTION 
# ---------------------------------------------------------
# Improve contrast and sharpness before detection.
def adaptive_enhance(gray):
    h, w = gray.shape[:2]
    enhanced = gray.copy()
    
    # Check frame brightness, contrast, and sharpness.
    mean_val = np.mean(gray)       
    std_val = np.std(gray)         
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var() 

    # Add contrast when the frame needs it.
    if mean_val < 90 or std_val < 40:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(enhanced)

    k_size = int(w * 0.005)
    k_size = k_size + 1 if k_size % 2 == 0 else k_size 
    k_size = max(3, k_size) 
    
    # Reduce noise while keeping edges.
    enhanced = cv2.bilateralFilter(enhanced, d=k_size, sigmaColor=25, sigmaSpace=25)

    if sharpness < 50:
        blur = cv2.GaussianBlur(enhanced, (3,3), 0)
        enhanced = cv2.addWeighted(enhanced, 1.5, blur, -0.5, 0)

    return enhanced

# ---------------------------------------------------------
# 4. ENHANCE EACH FRAME
# ---------------------------------------------------------
image_files = sorted([f for f in os.listdir(extract_frame_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

# Enhance each extracted frame.
for frame_name in image_files:
    img_path = os.path.join(extract_frame_dir, frame_name)
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    final_img = adaptive_enhance(gray)
    cv2.imwrite(os.path.join(enhanced_frame_dir, frame_name), final_img)
print("Frames enhanced.")

# ---------------------------------------------------------
# 5. SEGMENTATION FUNCTION (Cleaned to prevent giant blobs)
# ---------------------------------------------------------
# Build a mask for possible damage pixels.
def segment_frame(img):
    h, w = img.shape[:2]
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img
        
    mask = np.zeros_like(gray)

    # Keep only the road section.
    roi = np.array([[(int(w*0), int(h*1)), 
                     (int(w*0), int(h*0.5)),
                     (int(w*0.1), int(h*0.30)),
                     (int(w*0.9), int(h*0.30)), 
                     (int(w*1), int(h*0.5)),
                     (int(w*1), int(h*1))]], np.int32)
    cv2.fillPoly(mask, roi, 255)

    masked = cv2.bitwise_and(gray, mask)
    road_pixels = gray[mask == 255]

    if len(road_pixels) == 0: 
        return mask
        
    # Use road brightness to set the threshold.
    mu, sigma = np.mean(road_pixels), np.std(road_pixels)
    
    # Standard dynamic multiplier for stable detection
    dynamic_multiplier = np.clip(sigma / 25.0, 0.8, 1.8)
    dyn_thresh = int(mu - (dynamic_multiplier * sigma))
    
    dyn_thresh = max(10, min(180, dyn_thresh))

    # Make dark damage pixels white.
    _, binary = cv2.threshold(masked, dyn_thresh, 255, cv2.THRESH_BINARY_INV)
    binary = cv2.bitwise_and(binary, mask)
    
    # Mild mathematical blur to kill single-pixel noise without destroying cracks
    k_blur = max(3, int(w * 0.003))
    k_blur = k_blur + 1 if k_blur % 2 == 0 else k_blur
    binary = cv2.medianBlur(binary, k_blur) 
    
    return binary

# ---------------------------------------------------------
# 6. DAMAGE CLASSIFICATION FUNCTION (Aspect Ratio Filter)
# ---------------------------------------------------------
# Classify each damage region by shape.
def classify_damage(img, mask):
    h, w = img.shape[:2]
    total_pixels = h * w
    
    # Base Thresholds
    noise_threshold = total_pixels * 0.00003     # Extremely small limit
    small_crack_limit = total_pixels * 0.0003    # The boundary for "small" damage
    pothole_min_area = total_pixels * 0.0015     
    alligator_min_area = total_pixels * 0.001    

    # Find damage regions in the mask.
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detection = img.copy()
    overlay = img.copy()

    for cnt in contours:
        # Measure the region size.
        area = cv2.contourArea(cnt)
        
        # 1. Kill absolute microscopic noise
        if area < noise_threshold: continue 

        # 2. THE SHAPE FILTER: Calculate the Aspect Ratio (Length vs Width)
        # Measure the region shape.
        rect = cv2.minAreaRect(cnt)
        (width_rect, height_rect) = rect[1]
        
        # Prevent division by zero errors
        if width_rect == 0 or height_rect == 0: continue
        
        aspect_ratio = max(width_rect, height_rect) / min(width_rect, height_rect)

        # Logic: If it is small and round (ratio < 2.5), it is an asphalt pore/pebble. Ignore it.
        if area < small_crack_limit and aspect_ratio < 2.5:
            continue

        # --- Geometric Classification ---
        # Measure solidity and roundness.
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        solidity = area / hull_area if hull_area > 0 else 0
        h_perim = cv2.arcLength(hull, True)
        h_circ = (4 * np.pi * hull_area) / (h_perim**2) if h_perim > 0 else 0

        data = cnt.reshape(-1, 2).astype(np.float64)
        if len(data) > 3:
            mean, e_vecs, evals = cv2.PCACompute2(data, mean=None)
            ev1, ev2 = evals[0][0], (evals[1][0] if len(evals) > 1 else 0)
            spread_ratio = ev2 / ev1 if ev1 > 0 else 0
        else:
            spread_ratio = 0

        # Apply Classification Rules
        # Choose the damage type by shape.
        if area > pothole_min_area and h_circ > 0.50 and solidity > 0.40:
            color, draw_cnt = (0, 0, 255), hull # Pothole (Red)
        elif area > alligator_min_area and spread_ratio > 0.10 and solidity < 0.65:
            color, draw_cnt = (255, 0, 255), cnt # Alligator (Purple)
        else:
            color, draw_cnt = (255, 0, 0), cnt # Standard Crack (Blue)
            
        # Draw the colored result.
        cv2.drawContours(overlay, [draw_cnt], -1, color, -1)
        cv2.drawContours(detection, [draw_cnt], -1, color, 2)

    return cv2.addWeighted(overlay, 0.35, detection, 0.65, 0), mask

# ---------------------------------------------------------
# 7. PROCESSING FRAMES AND GENERATE OUTPUT VIDEOS
# ---------------------------------------------------------
print("Generating masks and final video outputs...")

frame_files = sorted([f for f in os.listdir(extract_frame_dir) if f.endswith(('.jpg', '.png'))])

sample_img = cv2.imread(os.path.join(extract_frame_dir, frame_files[0]))
h, w, _ = sample_img.shape
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

enhanced_video_writer = cv2.VideoWriter(enhanced_video_path, fourcc, 30.0, (w, h))
segmented_video_writer = cv2.VideoWriter(segmented_video_path, fourcc, 30.0, (w, h))
detected_video_writer = cv2.VideoWriter(detected_video_path, fourcc, 30.0, (w, h))
final_video_writer = cv2.VideoWriter(output_video_path, fourcc, 30.0, (w, h))

# Create masks, detections, and videos for each frame.
for frame_name in frame_files:
    img_orig = cv2.imread(os.path.join(extract_frame_dir, frame_name))
    img_enh_3channel = cv2.imread(os.path.join(enhanced_frame_dir, frame_name))
    
    enhanced_video_writer.write(img_enh_3channel) 
    
    mask = segment_frame(img_enh_3channel)
    cv2.imwrite(os.path.join(segmented_frame_dir, frame_name), mask) 
    segmented_video_writer.write(cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)) 
    
    detected_frame, _ = classify_damage(img_enh_3channel, mask)
    cv2.imwrite(os.path.join(detected_frame_dir, frame_name), detected_frame)
    detected_video_writer.write(detected_frame)
    
    final_frame, _ = classify_damage(img_orig, mask)
    cv2.imwrite(os.path.join(final_frame_dir, frame_name), final_frame)
    final_video_writer.write(final_frame)

# Save and close all output videos.
enhanced_video_writer.release()
segmented_video_writer.release()
detected_video_writer.release()
final_video_writer.release()

print("Process Complete! The aspect ratio filter is active.")

KeyboardInterrupt: 